In [2]:
import torch
from torch import nn
from torch.nn import functional as F

## BatchNorm dimension-agnostic from scratch
Notes:
- Use torch.var(unbiased=False) for exact replication of nn.BatchNorm
- $\mu$ and $\sigma^2$ are mean and variance of each channel across the batches of images' activation map, so each channels will be normalized by a different amount, but across images, same channels will be normalized by same amount
- $\gamma$ and $\beta$ are learning parameters for each channels

In [76]:
class BatchNorm(nn.Module):
    def __init__(self, num_channels, momentum=0.1, eps=1e-5):
        super().__init__()
        self.momentum = momentum
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(num_channels).float())
        self.beta = nn.Parameter(torch.zeros(num_channels).float())
        self.register_buffer('running_mean', torch.zeros(num_channels)) 
        self.register_buffer('running_var', torch.ones(num_channels))

    def forward(self, x): # (B, C, *)
        if self.training:
            mean = x.transpose(0, 1).mean(dim=list(range(1, x.ndim)), keepdim=True).transpose(0, 1) # (1, C, 1, 1)
            var = x.transpose(0, 1).var(dim=list(range(1, x.ndim)), keepdim=True, unbiased=True).transpose(0, 1) # (1, C, 1, 1)
            x = (x - mean) / torch.sqrt(var + self.eps) # transform the mean and var into (1, C, 1, 1) for broadcasting semanticj
            x = x * self.gamma.view([-1]+[1] * (x.ndim - 2)) + self.beta.view([-1]+[1] * (x.ndim - 2))
            self.running_mean = self.momentum * mean.view(-1) + (1-self.momentum) * self.running_mean
            self.running_var = self.momentum * var.view(-1) + (1-self.momentum) * self.running_var
        else:
            x = (x - self.running_mean.view([-1]+[1] * (x.ndim - 2))) / torch.sqrt(self.running_var.view([-1]+[1] * (x.ndim - 2)) + self.eps) # transform the mean and var into (1, C, 1, 1) for broadcasting semantic
            x = x * self.gamma.view([-1]+[1] * (x.ndim - 2)) + self.beta.view([-1]+[1] * (x.ndim - 2))
        return x

def check_bn(channels=32):
    x = torch.randn(10, channels, 16, 16)
    u = (BatchNorm(channels)(x) - nn.BatchNorm2d(channels)(x)).abs().max()
    x = torch.randn(10, channels, 16 * 16)
    v = (BatchNorm(channels)(x) - nn.BatchNorm1d(channels)(x)).abs().max()
    print("Batchnorm max mismatch values:", u.item(), v.item())

check_bn()

Batchnorm max mismatch values: 0.0008568763732910156 0.0009617805480957031


### Demonstrating BatchNorm's power with simple CNN architecture

In [4]:
device = 'cuda'
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
train_ds = datasets.MNIST(root="/tmp/data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="/tmp/data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self, norm2d, norm1d):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 2, 3, 1, 1),
            norm2d(2),
            nn.ReLU(),
            nn.AvgPool2d(2, 2),
            nn.Conv2d(2, 4, 3, 1, 1),
            norm2d(4),
            nn.ReLU(),
            nn.AvgPool2d(2, 2))
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(4 * 7 * 7, 20),
            norm1d(20),
            nn.ReLU(),
            nn.Linear(20, 10)
        )
    
    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

def train(model, epoch, lr):
    optimizer = torch.optim.SGD(model.parameters(), lr = lr)
    for _ in range(epoch):
        data_iter = iter(train_loader)
        total_loss = 0
        total_batches = 0 
        for img, label in data_iter:
            img, label = img.to(device), label.to(device)
            y = model(img)
            loss = F.cross_entropy(y, label)
            total_loss += loss.item()
            total_batches += 1
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(total_loss / total_batches)

def val(model):
    with torch.no_grad():
        data_iter = iter(test_loader)
        acc = 0
        total = 0
        for img, label in data_iter:
            img, label = img.to(device), label.to(device)
            y = model(img)
            acc += (y.argmax(dim=1, keepdim=False) == label).sum()
            total += len(label)
    print(acc.item()/total)

In [79]:
# Using implementation from scratch
model = SimpleCNN(norm2d=BatchNorm, norm1d=BatchNorm).to(device)
train(model, 1, 1)
val(model)
# Using torch.nn built-in
model = SimpleCNN(norm2d=nn.BatchNorm2d, norm1d=nn.BatchNorm1d).to(device)
train(model, 1, 1)
val(model)

0.1706372439188124
0.9733
0.1907513860041208
0.9719


## LayerNorm for Convolution from scratch
Notes:
- Use torch.var(unbiased=False) for exact replication of nn.LayerNorm
- $\mu$ and $\sigma^2$ are mean and variance of each point on the activation map across all channels, so all channels will be normalized by the same amount, but different images in the batch will be normalized by a different amount
- $\gamma$ and $\beta$ are still learning parameters for each channels

In [135]:
list(range(-2,0))

[-2, -1]

In [136]:
list(range(5 - 2))

[0, 1, 2]

In [142]:
type(1) is int

True

In [ ]:
variance_unbiased = False
class LayerNorm(nn.Module):
    VARIANCE_UNBIASED = False # False will make it identical to nn.LayerNorm
    def __init__(self, norm_shape, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.norm_ndim = 1 if type(norm_shape) is int else len(norm_shape)
        self.gamma = nn.Parameter(torch.ones(norm_shape))
        self.beta = nn.Parameter(torch.zeros(norm_shape)) 

    def forward(self, x):
        mean = torch.mean(x, dim=list(range(-self.norm_ndim, 0)), keepdim=True) # (*, *norm_shape) 
        var = torch.var(x, dim=list(range(-self.norm_ndim, 0)), keepdim=True, unbiased=self.VARIANCE_UNBIASED) # (*, *norm_shape)
        x = (x - mean) / torch.sqrt(var + self.eps)
        x = self.gamma * x - self.beta # x -> (*, *norm_shape); self.gamma -> (*norm_shape) will be broadcasted
        return x

class LayerNormConvLegacy(nn.Module):
    def __init__(self, num_channels, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(num_channels))
        self.beta = nn.Parameter(torch.zeros(num_channels))
    def forward(self, x): # (B, C, H, W)
        mean = torch.mean(x, dim=1, keepdim=True) # (B, 1, H, W)
        var = torch.var(x, dim=1, keepdim=True, unbiased=variance_unbiased) # (B, 1, H, W)
        x = (x - mean) / torch.sqrt(var + self.eps)
        x = self.gamma.view(1, -1, 1, 1) * x - self.beta.view(1, -1, 1, 1)
        return x
    
class LayerNormConv(nn.Module):
    def __init__(self, num_channels, eps=1e-5):
        super().__init__()
        self.norm = LayerNorm(num_channels, eps=eps)
    def forward(self, x):
        return self.norm(x.permute(0,2,3,1)).permute(0, 3, 1, 2)

class nnLayerNormConv(nn.Module):
    def __init__(self, num_channels, eps=1e-5):
        super().__init__()
        self.norm = nn.LayerNorm(num_channels, eps=eps)
    def forward(self, x):
        return self.norm(x.permute(0,2,3,1)).permute(0, 3, 1, 2)

def check_layer_norm(norm_shape=(20, 30, 40), eps=1e-5):
    x = torch.randn(10, 32, *norm_shape)
    norm0 = LayerNorm(norm_shape, eps)
    norm1 = nn.LayerNorm(norm_shape, eps)
    print("LayerNorm max mismatch:", (norm0(x) - norm1(x)).max())

def check_conv_norm(num_channels=32, eps=1e-5):
    x = torch.randn(10, num_channels, 16, 16)
    norm0 = LayerNormConv(num_channels, eps)
    norm1 = nnLayerNormConv(num_channels, eps)
    print("ConvLayerNorm max mismatch:", (norm0(x) - norm1(x)).max())

check_layer_norm(eps=1e-1)
check_conv_norm(eps=1e-2)

LayerNorm max mismatch: tensor(9.5367e-07, grad_fn=<MaxBackward1>)
ConvLayerNorm max mismatch: tensor(4.7684e-07, grad_fn=<MaxBackward1>)


Conclusion:
- LayerNorm is no good for convolution

In [166]:
model = SimpleCNN(norm2d=nn.Identity, norm1d=nn.Identity).to(device)
train(model, 1, 1)
val(model)


model = SimpleCNN(norm2d=BatchNorm, norm1d=LayerNorm).to(device)
train(model, 1, 1)
val(model)

model = SimpleCNN(norm2d=nn.Identity, norm1d=LayerNorm).to(device)
train(model, 1, 1)
val(model)

model = SimpleCNN(norm2d=LayerNormConv, norm1d=LayerNorm).to(device)
train(model, 1, 1)
val(model)

1.6908514082654198
0.1009
0.27621374655860476
0.9664
0.30782288421334614
0.9343
0.441589934763306
0.9241
